# PEFT - LoRA

https://huggingface.co/docs/peft/en/developer_guides/lora


**1. 다양한 초기화 전략 (Initialization)**

LoRA 가중치를 어떻게 시작하느냐에 따라 학습 속도와 성능이 달라진다.

* **기본값:** 가중치 A는 Kaiming-uniform, B는 0으로 초기화하여 처음에는 정등 변환(Identity transform) 상태로 시작한다.
* **PiSSA:** 주특이값(Principal singular values)을 사용하여 초기화하며, 일반 LoRA보다 수렴 속도가 빠르고 성능이 우수하다.
* **OLoRA:** QR 분해를 활용하여 베이스 모델의 가중치를 변환하며, 학습 안정성과 수렴 속도를 높인다.
* **EVA:** 입력 활성화 값에 대해 SVD를 수행하여 데이터 기반으로 초기화하며, 레이어별로 랭크(Rank)를 유연하게 할당한다.

**2. 양자화 모델 최적화 (LoftQ & DoRA)**

* **LoftQ:** 양자화된 모델을 미세 조정할 때 발생하는 오차를 최소화하도록 LoRA 가중치를 초기화하는 기술이다.
* **DoRA (Weight-Decomposed Low-Rank Adaptation):** 가중치 업데이트를 크기(Magnitude)와 방향(Direction)으로 분리하여 처리하며, 특히 낮은 랭크에서도 높은 성능을 보여준다.

**3. 효율적인 추론 및 학습 기법**

* **aLoRA (Activated LoRA):** 특정 토큰(invocation tokens)이 나타날 때만 어댑터를 활성화하는 방식이다. 기본 모델과 KV 캐시를 공유할 수 있어 추론 속도를 획기적으로 높일 수 있다.
* **Rank-stabilized LoRA (rsLoRA):** 랭크()의 제곱근에 비례하여 스케일링을 조절함으로써, 높은 랭크를 사용할 때 학습을 안정화시킨다.
* **레이어 복제 (Layer Replication):** 기존 모델의 레이어를 논리적으로 복제하여 모델 크기를 확장하되, 실제 메모리 사용량은 최소화하면서 어댑터를 추가하는 방식이다.

**4. 고급 최적화 및 제어**

* **특수 옵티마이저:** 가중치 A를 고정하고 B만 튜닝하여 메모리를 아끼는 **LoRA-FA**, A와 B에 서로 다른 학습률을 적용해 속도를 2배 높이는 **LoRA+** 등을 지원한다.
* **세부 제어:** 특정 레이어마다 서로 다른 랭크()나 알파() 값을 지정할 수 있는 기능을 제공한다.
* **토큰 학습:** 특정 레이어의 가중치뿐만 아니라, 특정 토큰 임베딩만 선택적으로 학습시키는 기능을 지원한다.

In [1]:
%pip install -Uqqq transformers datasets accelerate trl peft hf_transfer pydantic langchain-huggingface

Note: you may need to restart the kernel to use updated packages.


In [2]:
!nvidia-smi  # GPU 확인 (모델명 / GPU ID / 메모리 사용량)

'nvidia-smi'��(��) ���� �Ǵ� �ܺ� ����, ������ �� �ִ� ���α׷�, �Ǵ�
��ġ ������ �ƴմϴ�.


In [ ]:
# 로컬 기준 환경변수 설정
from dotenv import load_dotenv
import os

load_dotenv()
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
HF_TOKEN = os.getenv('HF_TOKEN')

In [ ]:
# Runpod 기준 환경변수 설정 (Pod에 환경변수가 있어야 함)
# import os

# OPENAI_API_KEY = os.environ('OPENAI_API_KEY')
# HF_TOKEN = os.environ('HF_TOKEN')

## 데이터셋 로드
https://huggingface.co/datasets/capybaraOh/naver-economy-news2stock

In [ ]:
from datasets import load_dataset  # HuggingFace 데이텃세 로더

# HuggingFace Hub train split 로드
dataset = load_dataset('capybaraOh/naver-economy-news2stock', split='train')
print(len(dataset))
dataset  # Dataset 객체 정보

In [ ]:
dataset[0]  # {'system': ..., 'user': ..., 'assistant': ...}

In [ ]:
# HuggingFace Dataset을 학습 / 평가 분리 후 Chat 메시지 포맷으로 변환
test_ratio = 0.2  # 평가셋 비율

train_data = []  # 학습 데이터 리스트
test_data = []   # 평가 데이터 리스트

data_indices = list(range(len(dataset)))  # 전체 인덱스
test_size = int(len(dataset) * test_ratio)  # 평가셋 크기

test_data_indices = data_indices[:test_ratio]   # 앞부분은 평가셋 (인덱스)
train_data_indices = data_indices[test_ratio:]  # 나머지는 학습셋 (인덱스)

# OpenAI / Chat 학습용 포맷 : {'messages': [{system}, {user}, {assistant}]}
def format_data(data):
    return {
        'messages': [
            {
                'role': 'system',
                'content': data['system']
            },
            {
                'role': 'user',
                'content': data['user']
            },
            {
                'role': 'assistant',
                'content': data['assistant']
            }
        ]
    }

train_data = [format_data(dataset[i]) for i in train_data_indices]  # 학습 인덱스 -> 학습 dict
test_data = [format_data(dataset[i]) for i in test_data_indices]    # 평가 인덱스 -> 평가 dict

print(len(train_data))
print(len(test_data))

In [ ]:
train_data[128]  # messages 포맷 dict 확인

In [ ]:
# List -> HuggingFace Dataset (내용은 그대로, 컨테이너만 변경)
from datasets import Dataset

train_dataset = Dataset.from_list(train_data)  # list -> Dataset
test_dataset = Dataset.from_list(test_data)    # list -> Dataset

train_dataset[128]

## NCSOFT/Llama-VARCO-8B-Instruct란?
https://huggingface.co/NCSOFT/Llama-VARCO-8B-Instruct


* **기반 모델:** Meta의 Llama-3.1-8B 모델을 기반으로 한다.
* **개발 목적:** 한국어 능력을 극대화하는 동시에 영어 구사 능력도 유지하도록 설계되었다.
* **학습 방법:** 한국어와 영어 데이터셋을 활용한 지속 사전 학습(Continual Pre-training)을 거쳤으며, 이후 지도 미세 조정(SFT)과 직접 선호도 최적화(DPO)를 통해 인간의 선호도에 맞게 정렬되었다.


**SFT에서 한국어능력향상과 동시에 영어능력유지란:**

일반적으로 한국어 데이터를 대량으로 추가 학습시키면 기존에 모델이 가지고 있던 영어 지식이 손상되는 '파괴적 망각(Catastrophic Forgetting)' 현상이 발생한다. 엔씨소프트는 이를 방지하기 위해 **지속 사전 학습(Continual Pre-training)**을 적용했다.

**_1. 데이터 믹스(Data Mixing) 전략:_**

단순히 한국어 데이터만 밀어 넣는 것이 아니라, 모델이 이미 학습했던 영어 데이터와 고품질의 한국어 데이터를 특정 비율로 섞어 학습한다. 이를 통해 기존의 영어 추론 능력을 '복습'하면서 새로운 언어 체계를 '습득'하게 된다.

**_2. 토크나이저 효율화와 임베딩 확장:_**

기존 Llama-3.1의 토크나이저 성능을 유지하면서 한국어 표현력을 높이기 위해 어휘 사전(Vocabulary)을 최적화한다. 영어 토큰 정보는 건드리지 않고 한국어 토큰의 밀도를 높여 두 언어 간의 연결 고리를 강화하는 방식이다.

**_3. 지식 전이(Knowledge Transfer):_**

영어 데이터로 학습된 모델의 강력한 논리적 사고 능력을 한국어로 전이시키는 과정을 거친다.

* **추론 능력 유지:** 수학이나 코딩 같은 논리적 작업은 영어 데이터에서 배운 구조를 그대로 활용한다.
* **언어 정렬:** SFT(지도 미세 조정) 단계에서 동일한 질문을 한국어와 영어로 번급하며 학습시켜, 언어에 상관없이 일관된 답변을 내놓도록 유도한다.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM  # 토크나이저 / 생성형 모델 자동 로더
import torch

pretrained_model_name = 'NCSOFT/Llama-VARCO-8B-Instruct'  # 사전학습 모델명

model = AutoModelForCausalLM.from_pretrained(
    pretrained_model_name,
    dtype = torch.bfloat16,  # 가중치 로딩 dtype(bf16)
    device_map = 'auto'      # 환경에 맞춰 CPU / GPU 자동 배치
)

tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name)  # 해당 모델의 토크나이저 로드

## llama-3 chat template 변환

Llama3 모델은 특정 chat template 형식으로 학습되어, 그 형식을 사용해야 최적 성능을 낼 수 있다.
Chat template을 사용하지 않으면 모델이 대화 구조를 제대로 인식하지 못할 수 있다.
opean_ai 형식의 데이터를 llama-3 형식으로 변환한다.


**LLaMA-3 채팅 포맷**
LLaMA-3 채팅 포맷은 LLaMA-3 계열 챗봇 모델이 대화 내용을 이해하고 답변할 수 있도록 만들어진 입력 데이터 구조입니다.
여러 역할(시스템, 유저, 어시스턴트)의 메시지를 특별한 토큰과 구조로 묶어서 하나의 프롬프트로 합치는 방식입니다.
구조 예시
아래와 같이 대화 흐름을 명확히 구분하는 토큰들이 사용됩니다:

```
<|begin_of_text|>
<|start_header_id|>system<|end_header_id|>
[시스템 역할 지침]<|eot_id|>
<|start_header_id|>user<|end_header_id|>
[유저 질문]<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>
[모델의 답변]<|eot_id|>
```
* <|begin_of_text|> : 전체 프롬프트의 시작을 알리는 토큰
* <|start_header_id|>role<|end_header_id|> : 각 메시지의 역할 구분(시스템, 유저, 어시스턴트 등)
* 각 메시지 끝에 <|eot_id|> : 하나의 메시지 블록이 끝났음을 알림
* 마지막 assistant 블럭은 응답 생성 위치를 가리킨다. apply_chat_template(add_generation_prompt=False)로 설정했더라도 내부 템플릿에는 응답을 받을 자리 표시자로 <|assistant|> 토큰이 남아 있어, "여기서부터 어시스턴트가 답변을 생성해야 한다"는 신호를 제공하는 것임.

**왜 이 포맷이 필요할까?**

* 모델이 **“어디까지가 시스템 안내, 어디서부터가 유저 질문, 어디서부터가 답변인지”** 정확하게 파악할 수 있다.
* 여러 턴(turn)의 대화가 이어질 때도 메시지 경계를 명확히 구분해 혼동 없이 맥락을 유지할 수 있다.
* LLaMA-3 계열 모델은 이런 포맷으로 학습되어 있기 때문에 **실전 파인튜닝/추론 시에도 반드시 이 구조로 입력해야** 기대하는 챗봇 성능을 발휘할 수 있다.

In [ ]:
# 하나의 샘플만 openai 방식 메시지 -> llama3 방식 메시지로 변환
text = tokenizer.apply_chat_template(train_dataset[128]['messages'], tokenize=False)
print(text)

### data_collator 함수

* 미니배치(batch) 데이터를 모델이 바로 학습할 수 있는 형태(토큰·마스크·정답)로 변환합니다.
* 특히 아래와 같은 LLaMA-3 채팅 포맷을 쓸 때,
  “어디까지가 질문/어디서부터가 답변(assistant)인지”를 정확히 구분해서
  모델이 정답(답변 부분)만 학습하도록 레이블을 지정합니다.

#### 함수 설명

**1. 프롬프트 생성 (Prompt Construction)**

입력받은 `batch` 데이터는 리스트 내에 여러 메시지(`system`, `user`, `assistant`)를 포함하는 사전(dict) 구조이다.

* Llama 3의 특수 토큰(` <|begin_of_text|>`, `<|start_header_id|>`, `<|eot_id|>`)을 사용하여 모든 대화 내용을 하나의 긴 문자열로 병합한다.
* 각 역할(role)의 시작과 끝을 명확히 구분하여 모델이 대화 맥락을 이해할 수 있도록 구성한다.

**2. 토크나이즈 및 패딩 (Tokenization)**

병합된 문자열 리스트를 `tokenizer`를 통해 숫자 ID(`input_ids`)로 변환한다.

* `padding=True`: 배치 내의 문장들 중 가장 긴 문장을 기준으로 길이를 맞춘다.
* `truncation=True`: `max_length`를 초과하는 데이터는 절단한다.
* `return_tensors="pt"`: PyTorch 텐서 형식으로 결과를 반환한다.

**3. 레이블 생성 및 Loss Masking**

이 함수의 핵심 부분이다. 모델이 '사용자의 질문'이 아닌 **'모델의 답변(assistant)'** 부분에 대해서만 학습하도록 설정한다.

* **-100 값의 의미**: PyTorch의 `CrossEntropyLoss`는 레이블 값이 `-100`인 경우 손실(Loss) 계산에서 제외한다. 이를 통해 모델은 질문 부분을 예측하려고 노력하지 않고, 답변 부분의 정확도에만 집중하게 된다.
* **구간 탐색**: `assistant_tokens`를 기점으로 답변이 시작되는 위치를 찾고, `<|eot_id|>` 토큰이 나오는 지점까지의 인덱스를 추출한다.
* **값 복사**: 해당 구간의 `labels`에만 실제 `input_ids` 값을 복사하여 넣는다.

In [ ]:
# Lllam3 계열 Chat 학습용 데이터 콜레이터 : 프롬프트 생성 -> 토크나이즈/패딩 -> assistant 구간만 라벨링
def data_collator(batch, tokenizer=tokenizer, max_length=8192):
    # 1. 프롬프트 생성
    prompts = []
    for example in batch:
        prompt = '<|begin_of_text|>'
        for msg in example['messages']:
            role = msg['role']
            content = msg['content'].strip()
            prompt += f'<|start_header_id|>{role}<|end_header_id|>\n{content}<|eot_id|>'
        prompts.append(prompt)
    print(prompt)

    # 2. 토큰처리 / 패딩 / 텐서 변환
    tokenized = tokenizer(
        prompt,
        truncation=True,
        max_length=max_length,
        padding=True,
        return_tensors='pt'
    )
    input_ids = tokenized['input_ids']
    attention_mask = tokenized['attention_mask']
    print(tokenized)
    print(len(tokenized['input_ids'][0]))
    print(len(tokenized['input_ids'][1]))
    print(len(tokenized['attention_mask'][0]), len(tokenized['attention_mask'][0].sum().item()))
    print(len(tokenized['attention_mask'][0]), len(tokenized['attention_mask'][0].sum().item()))

    # 3. 라벨 생성
    labels = torch.full_like(input_ids, fill_value=-100)
    print(labels.shape)

    assistant_header = '<|start_header_id|>assistant<|end_header_id|>\n'
    assistant_token_id = tokenizer.encode(assistant_header, add_special_tokens=False)
    eot_token = '<|eot_id|>'
    eot_token_id = tokenizer.encode(eot_token, add_special_tokens=False)
    print(assistant_token_id)
    print(eot_token_id)

    for i, ids in enumerate(input_ids):
        ids_list = ids.tolist()
        start = None
        for idx in range(len(ids_list) - len(assistant_token_id) + 1):
            if ids_list[idx:idx+len(assistant_token_id)] == assistant_token_id:
                start = idx + len(assistant_token_id)
                break

        if start is not None:
            end = None
            for idx in range(start, len(ids_list - len(eot_token_id) + 1)):
                if ids_list[idx:idx+len(eot_token_id)] == eot_token_id:
                    end = idx + len(eot_token_id)
                    break

        labels[i, start:end] = input_ids[i, start:end]

    return {
        'input_ids': input_ids,
        'attention_mask': attention_mask,
        'labels': labels
    }

data_collator([train_dataset[0], train_dataset[1]])

NameError: name 'tokenizer' is not defined

## PEFT Finetuning - LoRA

* LoRA는 **"Low-Rank Adapter(저랭크 어댑터)"**
* 거대한 대형언어모델(LLM)의 **전체 파라미터를 일일이 미세조정(파인튜닝)하지 않고**,
  **딱 필요한 핵심 부분만 저렴하게 빠르게 학습**하는 최신 파인튜닝.
* **"LLM의 성능은 그대로, 비용/시간/메모리/유지보수는 최소로"** 파인튜닝을 할 수 있게 해주는 AI 실무에서 가장 중요한 기법 중 하나이다.

**왜 LoRA가 등장했을까?**

* GPT, Llama, DeepSeek 같은 대형언어모델은 **파라미터(매개변수) 수가 수십억\~수조 개**나 된다.
* 이런 모델을 파인튜닝하려면 **막대한 GPU 메모리와 시간, 저장 공간**이 필요.
* 하지만, 실제로 특정 태스크에 맞게 모델을 조정할 때 **전체를 다 바꿀 필요가 없다.**
* 대부분의 정보는 기존 모델에 이미 들어있고,
  **특정 입력(질문)과 특정 출력(답변)의 관계만 살짝 조정**해주면 충분하다.

**LoRA의 원리**

* 기존 대형 모델의 핵심 연산(주로 "곱셈" 부분)에
  **작고 얇은 "보조 네트워크(어댑터 레이어)"**를 덧붙인다.
* 전체 모델은 거의 건드리지 않고,
  **이 어댑터 레이어의 파라미터만 새로 추가해서 학습**
* 학습이 끝나면,

  * 원본 모델은 그대로
  * 어댑터(작은 추가 파라미터)만 별도로 저장하면 끝!
* 추론할 땐 **원본 모델 + LoRA 어댑터**를 합쳐서 쓸 수 있다.

**LoRA의 장점**

* **파인튜닝 비용(시간, 메모리, 저장 용량)이 압도적으로 절약**된다.
* 7B, 13B, 70B 등 대형 모델도
  **일반 GPU(24GB/48GB)로도 쉽게 파인튜닝**이 가능하다.
* **동일한 원본 모델에 다양한 LoRA 어댑터만 바꿔 끼우며
  다양한 분야별 파인튜닝 결과를 쉽게 쓸 수 있다.**

**LoRA와 기존 방식의 비교**

* **기존 파인튜닝:**
  전체 파라미터(수십\~수백 GB)를 새로 저장/관리/학습 → 비효율적
* **LoRA:**
  원본은 그대로 두고,
  변화가 필요한 부분(수 MB\~수십 MB)만 별도로 학습/저장


**실전에서의 활용 예시**

* 번역 LoRA, 요약 LoRA, 감정분석 LoRA 등
  **하나의 원본 모델에 여러 용도별 어댑터를 저장/관리**할 수 있다.
* **A100 80GB, 3090, T4 등 다양한 GPU 환경에서도
  고성능 LLM 튜닝이 매우 쉽게 가능하다.**

In [6]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    lora_dropout=0.1,
    bias='none',
    target_modules=['q_proj, v_proj'],
    task_type='CAUSAL_LM'
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

NameError: name 'model' is not defined

In [7]:
from trl import SFTConfig  # TRL SFT 학습 설정 클래스

hub_model_id = 'capybaraOh/Llama-VARCO-8b-news2stock-analyzer'  # 학습 완료 후 업로드할 Hub 모델 ID

sft_config = SFTConfig(  # SFT 학습 하이퍼파라미터/저장/로그 설정
    output_dir="Llama-VARCO-8b-news2stock-analyzer", # 학습 완료된 모델과 체크포인트가 저장될 경로이다.
    num_train_epochs=3,                              # 전체 데이터셋을 반복 학습할 횟수(Epoch)이다.
    per_device_train_batch_size=2,                   # 각 GPU(장치)당 한 번에 처리할 데이터 샘플의 개수이다.
    gradient_accumulation_steps=2,                   # 그래디언트를 2번 누적한 후 가중치를 업데이트한다. (실제 배치 크기 = 2 * 2 = 4 효과를 낸다.)
    gradient_checkpointing=True,                     # VRAM 절약을 위해 중간 활성화 값을 저장하지 않고 역전파 시 재계산하는 설정이다.
    optim="adamw_torch_fused",                       # 최적화 알고리즘 설정이다. fused 버전은 CUDA에서 더 빠르다.
    logging_steps=10,                                # 10 스텝마다 학습 로그(Loss 등)를 출력한다.
    save_strategy="steps",                           # 체크포인트 저장 기준을 'steps'(스텝 수)로 설정한다. (옵션: 'epoch')
    save_steps=50,                                   # 50 스텝마다 모델 체크포인트를 저장한다.
    bf16=True,                                       # BF16(Brain Float 16) 정밀도를 사용하여 메모리를 아끼고 연산 속도를 높인다. (Ampere GPU 이상 권장)
    learning_rate=1e-4,                              # 학습률(Learning Rate)이다. 가중치 업데이트의 크기를 결정한다.
    max_grad_norm=0.3,                               # 그래디언트 클리핑 임계값이다. 그래디언트 폭주를 막아 학습 안정성을 높인다.
    warmup_ratio=0.03,                               # 전체 학습 단계의 3% 동안 학습률을 서서히 올리는 웜업(Warmup)을 수행한다.
    lr_scheduler_type="constant",                    # 학습률 스케줄러 타입이다. 여기서는 학습률을 변동 없이 상수로 유지한다.
    push_to_hub=True,                                # 학습이 끝나면 Hugging Face Hub에 모델을 자동으로 업로드한다.
    hub_model_id=hub_model_id,                       # Hub에 업로드될 때 사용될 저장소(Repository) ID이다.
    hub_token=True,                                  # Hub 업로드를 위해 인증 토큰을 사용한다.
    remove_unused_columns=False,                     # 데이터셋에서 모델의 forward 메서드 시그니처에 없는 컬럼을 자동으로 삭제하지 않도록 한다.
    dataset_kwargs={"skip_prepare_dataset": True},   # 데이터셋 처리 과정(packing 등)을 건너뛰도록 하는 설정이다.
    report_to=[],                                    # 학습 기록을 전송할 툴(WandB, Tensorboard 등)을 지정한다. 빈 리스트는 기록하지 않음을 의미한다.
    label_names=["labels"],                          # 손실(Loss) 계산 시 정답(Target)으로 사용할 데이터셋의 컬럼 이름이다.
)

TypeError: SFTConfig.__init__() got an unexpected keyword argument 'warmup_ratio'

In [8]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    ars=sft_config,
    train_dataset=train_dataset,
    data_collator=data_collator
)

trainer.train()

NameError: name 'model' is not defined

In [9]:
prompt_list = []
label_list = []

for messages in test_dataset['messages']:
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_gemeration_prompt=False)
    input = text.split('<|start_header_id|>assistant<|end_header_id|>\n')[0] + \
        '<|start_header_id|>assistant<|end_header_id|>\n'
    labels = text.split('<|start_header_id|>assistant<|end_header_id|>\n')[1].split('<|eot_id|>')[0]
    prompt_list.append(input)
    label_list.append(labels)

NameError: name 'test_dataset' is not defined

In [10]:
prompt_list[100]

IndexError: list index out of range

In [11]:
label_list[100]

IndexError: list index out of range

### 추론모델 - 런타임결합
1. lora모델을 로드
2. base모델 + lora adapter 세팅

In [ ]:
from transformers import AutoTokenizer, pipeline
import torch
from peft import AutoPeftModelForCausalLM


peft_model_name = 'NCSOFT/Llama-VARCO-8B-Instruct'

finetuned_model = AutoPeftModelForCausalLM.from_pretrained(
    pretrained_model_name,
    dtype=torch.bfloat16,
    device_map='auto'
)

tokenizer = AutoTokenizer.from_pretrained(peft_model_name)
pipe = pipeline('text-generation', model=finetuned_model, tokenizer=tokenizer)
pipe

In [ ]:
eos_token = tokenizer('<|eot_id|>', add_special_tokens=False)['input_ids'][0]
eos_token

In [ ]:
def test_inference(pipe, prompt):
    outputs = pipe(prompt, max_new_tokens=1024, eos_token_id=eos_token, do_sample=False)
    assistant_start = len(prompt)
    return outputs[0]['generated_text'][assistant_start:].strip()

for prompt, label in zip(prompt_list[10:13], label_list[10:13]):
    print(f'[prompt] : {prompt}')
    print(f'[label] : {label}')
    print(f'[response] : {test_inference(pipe, prompt)}')
    print('=' * 100)

    

In [12]:
def inference(news):
    messages = [
        {'role': 'system', 'content': '''
        당신은 금융/경제 뉴스의 핵심내용을 요약해 설명하고,
        특정 상장 종목에 미치는 긍정/부정 영향여부, 이유, 근거를 분석하는 금융/경제 분석 전문가입니다.

        다음 출력지시사항을 지켜주세요.
        1. 뉴스와 종목간의 연관성을 발견할 수 없다면:
            - stock_related를 False로 작성하세요.
            - summary에 뉴스의 요약을 작성하세요.
        2. 뉴스와 종목간의 연관성을 발견했다면:
            - stock_related를 True로 작성하세요.
            - summary에 뉴스의 요약을 작성하세요.
            - 긍정영향이 예상되는 종목이 있다면, positive_stocks, positive_keywords, positive_reasons를 작성하세요.
            - 부정영향이 예상되는 종목이 있다면, negative_stocks, negative_keywords, negative_reasons를 작성하세요.
            - 값이 없는 경우 빈 문자열(''), 빈 리스트([])로 작성하세요.
        '''},
        {'role': 'user', 'content': news}
    ]

    promtpt = tokenizer.apply_chat_template(messages, tokenize=False)
    outputs = pipe(prompt, max_new_token=1024, eos_token_id=eos_token, do_sample=False)
    assistant_start = len(prompt)
    return outputs[0]['generated_text'][assistant_start:].strip()

In [13]:
news = '''
7000피 환호도 잠시… 중동 불꽃에 '와르르'

코스피가 8일 장중 7000선을 돌파했으나 정규장 막판 지정학적 리스크 등 매크로 악재가 부각되며 하락 전환했다.

8일 한국거래소에 따르면 이날 코스피 지수는 전 거래일 대비 40.87포인트(0.58%) 내린 6954.52로 거래를 마쳤다. 4거래일 만의 하락세다. 지수는 전장보다 50.40포인트(0.72%) 오른 7045.79로 출발해 장중 7171.52까지 상승했으나, 장 후반 동력이 약화되며 전강후약 흐름을 보였다. 코스피가 장중 7000을 되찾은 건 지난달 18일 이후 15거래일 만이다.

이날 외국인과 기관은 각각 6616억원, 6495억원어치를 순매수하며 4거래일 연속 '쌍끌이' 매수를 이어갔다. 반면 개인은 3조533억원 규모를 순매도하며 매도 우위를 보였다.

증시는 초반 오픈AI발 호재로 대형 반도체주가 강세를 보이며 상승 출발했다. 하지만 오후 들어 중동 지역의 지정학적 불안이 고조되면서 투자 심리가 급격히 냉각됐다. 사우디아라비아 남부 에너지 시설이 예멘 반군의 공격을 받아 일부 가동이 중단됐다는 소식이 전해지자 국제 유가가 급등했다. 서부텍사스산원유(WTI) 선물은 배럴당 94달러, 브렌트유는 98달러를 돌파했으며 미국 국채 10년물 금리도 4.8%대로 상승했다.

여기에 오는 10일과 11일 발표를 앞둔 미국의 생산자물가지수(PPI)와 소비자물가지수(CPI)에 대한 경계감, 미·캐나다 간 관세 갈등, 엔화 강세에 따른 엔 캐리 트레이드 청산 우려 등이 복합적으로 작용된 것으로 보인다.

주요 종목 중 삼성전자는 한때 27만9000원까지 올랐지만 약보합세로 돌아서 26만9500원(-0.19%)에 마감했고, SK하이닉스도 장중 최고가 188만8000원에서 상승 폭을 대부분 반납한 179만3000원(0.56%)에 거래를 마쳤다. 삼성전기(-5.78%), LG에너지솔루션(-3.86%), 현대차(-2.04%) 등 시가총액 상위주 전반이 내림세를 보였다.

코스닥 지수 역시 전 거래일보다 10.31포인트(1.25%) 내린 811.88에 장을 마감했다. 외국인과 개인이 각각 607억 원, 1490억 원을 순매수했으나 기관이 2178억 원을 순매도하며 지수를 끌어내렸다.

이날 오후3시 30분 기준 서울 외환시장에서 원/달러 환율은 전일 대비 5.1원 오른 1345.6원에 거래됐다.
'''

inference(news)

NameError: name 'tokenizer' is not defined

In [ ]:
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    dtype=torch.bfloat16,
    device_map='auto'
)

base_pipe = pipeline('text-generation', model=base_model, tokenizer=tokenzier)

for idx, (prompt, label) in enumerate(zip(prompt_list[10:13], label_list[10:13])):
    print(f'[샘플 {idx+1}]')
    base_resp = test_inference(base_pipe, prompt)
    lora_resp = test_inference(pipe, prompt)
    print(f'[Base - 파인튜닝 전] {base_resp}')
    print(f'[LoRA - 파인튜닝 후] {lora_resp}')
    print(f'[Label] {label}')
    print('=' * 100)

In [ ]:
import os  # 환경변수(HF_TOKEN) 사용
from huggingface_hub import HfApi  # Hub API 사용

repo_id = "kty2001/Llama-VARCO-8b-news2stock-analyzer"  # 업로드할 모델 repo id
local_dir = "./Llama-VARCO-8b-news2stock-analyzer"  # 로컬 모델 폴더 경로

api = HfApi(token=os.environ["HF_TOKEN"])  # Hub 인증 토큰으로 API 객체 생성
api.create_repo(repo_id=repo_id, repo_type="model", exist_ok=True)  # repo가 없으면 생성(있으면 그대로 사용)

api.upload_folder(  # 로컬 폴더 전체를 Hub로 업로드
    folder_path=local_dir,  # 업로드할 로컬 폴더
    repo_id=repo_id,  # 대상 repo
    repo_type="model",  # 모델 repo로 업로드
    ignore_patterns=["checkpoint-*", "**/checkpoint-*"],  # 체크포인트 폴더는 제외
    commit_message="Upload final LoRA adapter (without checkpoints)"  # 커밋 메시지
)